# Week 3 — Context Engineering II

**AI Agentic Engineering · Corte 1**

Companion notebook to `week-03-context-engineering-ii-content.html`.

**You will practice:**
1. Sliding-window truncation vs. rolling summarization on a synthetic conversation.
2. Generating embeddings and computing cosine similarity.
3. A minimal semantic search function (the seed of next week's RAG system).
4. A light ADK / LangChain preview of the summarization call.
5. Two open exercises.


In [ ]:
%pip install -q --upgrade google-genai google-adk langchain-google-genai langgraph python-dotenv numpy pydantic

In [ ]:
import os
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv()
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-flash-latest"
EMBED_MODEL = "gemini-embedding-001"

## 1. Sliding window vs. rolling summarization

In [ ]:
history = [
    {"role": "user", "content": "Hi, I'm planning a trip to Peru."},
    {"role": "model", "content": "Great! When are you thinking of traveling, and for how long?"},
    {"role": "user", "content": "Sometime in August, for about 2 weeks."},
    {"role": "model", "content": "August is dry season in the Andes, good for Machu Picchu. Any budget in mind?"},
    {"role": "user", "content": "Around $2000 total, excluding flights."},
    {"role": "model", "content": "That's workable for hostels and local transport. Interested in the Amazon too, or just the highlands?"},
    {"role": "user", "content": "Just the highlands. I don't do well with humidity."},
    {"role": "model", "content": "Noted — highlands only. Cusco, Sacred Valley, and Machu Picchu it is."},
    {"role": "user", "content": "Also I'm vegetarian, will that be an issue?"},
    {"role": "model", "content": "Not at all, Peruvian highland cuisine has plenty of vegetarian options."},
]

def sliding_window(history, max_turns=4):
    return history[-max_turns:]

print("--- Sliding window (last 4) ---")
for m in sliding_window(history, max_turns=4):
    print(m["role"], ":", m["content"])

In [ ]:
def summarize_history(history):
    transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history)
    prompt = (
        "Summarize the key facts, decisions, and constraints from this conversation "
        "in 3-5 bullet points. Be specific.\n\n" + transcript
    )
    response = client.models.generate_content(model=MODEL, contents=prompt)
    return response.text

def compact_context(history, keep_recent=4):
    if len(history) <= keep_recent:
        return history
    older, recent = history[:-keep_recent], history[-keep_recent:]
    summary = summarize_history(older)
    return [{"role": "system", "content": f"Conversation summary so far:\n{summary}"}] + recent

compacted = compact_context(history, keep_recent=4)
print("--- Rolling summarization + last 4 ---")
for m in compacted:
    print(m["role"], ":", m["content"])

Compare the two outputs above. Sliding window silently drops "vegetarian" and "highlands only, no humidity"
if the window is small enough — summarization keeps them. Try lowering `max_turns` to 2 and see which strategy
still remembers the vegetarian constraint.

## 2. Embeddings and cosine similarity

In [ ]:
def embed(text):
    result = client.models.embed_content(model=EMBED_MODEL, contents=text)
    return result.embeddings[0].values

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

v_brake = embed("The bike's front brake feels loose.")
v_caliper = embed("The caliper on the front wheel isn't gripping properly.")
v_weather = embed("It might rain this weekend.")

print("brake vs caliper (related): ", round(cosine_similarity(v_brake, v_caliper), 3))
print("brake vs weather (unrelated):", round(cosine_similarity(v_brake, v_weather), 3))

## 3. A minimal semantic search function

In [ ]:
candidates = [
    "To fix a loose brake, tighten the caliper bolts and check the brake pads for wear.",
    "Flat tires are usually caused by punctures or worn-out inner tubes.",
    "Our rental bikes come in small, medium, and large frame sizes.",
    "Helmets are provided free of charge with every rental.",
    "The gear shifter may need cable tension adjustment if shifting feels sluggish.",
    "Rentals can be extended by messaging support at least 2 hours before the due time.",
]
candidate_vectors = [embed(c) for c in candidates]

def semantic_search(query, top_k=3):
    q_vec = embed(query)
    scored = [(cosine_similarity(q_vec, v), text) for v, text in zip(candidate_vectors, candidates)]
    scored.sort(reverse=True)
    return scored[:top_k]

for score, text in semantic_search("my brake feels wobbly and doesn't stop well"):
    print(f"{score:.3f}  {text}")

Notice the top result shares almost no exact words with the query ("wobbly", "doesn't stop well" vs.
"loose", "caliper bolts") — that's semantic search working as intended. Next week we scale this exact pattern
into a full RAG pipeline with chunking and a real vector database.

## 4. Light preview: the summarization call via ADK and LangChain

Same idea as previous weeks — reproducing one call (this time, `summarize_history`) through different tools.

In [ ]:
import asyncio
from google.adk.agents import Agent
from google.adk.runners import InMemoryRunner

summarizer_agent = Agent(
    model=MODEL,
    name="summarizer_agent",
    instruction="Summarize the given conversation in 3-5 specific bullet points.",
)

def ask_adk_agent(agent, prompt, app_name="week3_app", user_id="student"):
    runner = InMemoryRunner(agent=agent, app_name=app_name)
    session = asyncio.run(runner.session_service.create_session(app_name=app_name, user_id=user_id))
    content = types.Content(role="user", parts=[types.Part.from_text(text=prompt)])
    final_text = None
    for event in runner.run(user_id=user_id, session_id=session.id, new_message=content):
        if event.content and event.content.parts and event.content.parts[0].text:
            final_text = event.content.parts[0].text
    return final_text

transcript = "\n".join(f"{m['role']}: {m['content']}" for m in history[:-4])
print("ADK summary:\n", ask_adk_agent(summarizer_agent, transcript))

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(model=MODEL)
lc_summary = llm.invoke(
    "Summarize the key facts, decisions, and constraints from this conversation in 3-5 bullet points:\n\n"
    + transcript
)
print("LangChain summary:\n", lc_summary.content)

# LangChain also wraps the embedding model the same way:
lc_embeddings = GoogleGenerativeAIEmbeddings(model=f"models/{EMBED_MODEL}")
lc_vector = lc_embeddings.embed_query("The bike's front brake feels loose.")
print("\nLangChain embedding dims:", len(lc_vector))

## 5. Exercises

In [ ]:
# TODO Exercise 1 — Ranked semantic search with scores
# Extend `semantic_search` (or write a new version) so it also accepts a `threshold` parameter
# and drops any result below that similarity score, in addition to returning top_k.
# Test it with a query that has no good match in `candidates` and confirm it returns nothing (or few) results.

# your code here


In [ ]:
# TODO Exercise 2 — Structured memory extraction
# Define a Pydantic model `TripMemory` with fields like `destination`, `month`, `budget_usd`,
# `dietary_restrictions: list[str]`, `regions_of_interest: list[str]`.
# Use schema-forced JSON output (from Week 2) to extract a TripMemory instance from the
# `history` conversation above, instead of a paragraph summary.

class TripMemory(BaseModel):
    ...  # your fields here

# your generate_content call here


## Next week

Week 4 — **Retrieval-Augmented Generation (RAG)**: chunking strategies, ChromaDB and FAISS, and building a
complete, functional RAG system — the foundation of your Corte 1 project. See `week-04-rag-content.html`.